# LangGraph G13 — Streaming, tracing and evaluation
Three operational abilities remain. **Streaming** shows progress while the graph runs, so a
user is not staring at a spinner. **Tracing** records what ran, in what order, with what
arguments: it is how you answer "why did it register this student?". **Evaluation** replaces
"it worked when I tried it" with a set of cases and a pass rate, so a change to a prompt, a
tool description or a model can be judged before it ships.

Agent evaluation differs from ordinary testing because the output is a *trajectory*, not a
single value. Useful checks: was the right desk chosen? were the right tools called? was the
answer grounded? how many model calls and how long did it take?

### Step 1 — Stream node updates and model tokens

`stream(stream_mode="updates")` yields one item per finished node; `stream_mode="messages"`
yields model tokens with the name of the node producing them. Both work on every graph above.

In [ ]:
print("UPDATES (one per node):")
for update in campusai_v7.stream({"messages": [HumanMessage("Is CS201 open and what are its prerequisites?")]}, stream_mode="updates"):   # LangGraph
    for node, payload in update.items():
        last = payload.get("messages", [None])[-1] if isinstance(payload, dict) and payload.get("messages") else None
        detail = (", ".join(c["name"] for c in last.tool_calls) if isinstance(last, AIMessage) and last.tool_calls else text_of(last)[:60]) if last else str(payload)[:60]
        print(f"  {node:10} -> {detail}")

print("\nTOKENS from the model node:")
for token, metadata in campusai_v1.stream({"messages": [HumanMessage("Hello, what can you do?")]}, stream_mode="messages"):   # LangGraph: (AIMessageChunk, metadata)
    if text_of(token):
        print(text_of(token), end="", flush=True)
print("\n(node:", metadata.get("langgraph_node"), ")")

### Step 2 — A trace from the stream, with timings and token counts

The same `updates` stream, recorded instead of printed, is a trace: node, duration, tool calls,
tokens. Hosted tracing products (LangSmith is the one built for LangGraph; set
`LANGSMITH_TRACING=true` and an API key) record exactly this for every run without code changes.

In [ ]:
def traced_run(graph, question, **kwargs):                 # ours: run a graph and return (final state, trace rows)
    rows, started, tokens = [], time.perf_counter(), 0
    final_state = None
    for update in graph.stream({"messages": [HumanMessage(question)]}, stream_mode="updates", **kwargs):   # LangGraph
        for node, payload in update.items():
            messages = payload.get("messages", []) if isinstance(payload, dict) else []
            for m in messages:
                if isinstance(m, AIMessage) and m.usage_metadata:
                    tokens += m.usage_metadata.get("total_tokens", 0)
            calls = [c["name"] for m in messages if isinstance(m, AIMessage) for c in m.tool_calls]
            rows.append((node, round(1000 * (time.perf_counter() - started)), calls))
    return rows, tokens

rows, tokens = traced_run(campusai_v7, "Student S001 has 68% attendance; can they sit the CS201 exam?")
print("TRACE (node, ms since start, tool calls requested):")
for row in rows:
    print("  ", row)
print("tokens used:", tokens)

### Step 3 — An evaluation set and a pass rate

Each case states the question, the desk that should handle it, and the tools that must be
called. The harness runs the graph, compares, and reports a pass rate with cost and latency.
Keep such a set in version control and run it whenever a prompt, a tool or the model changes.

In [ ]:
EVAL_CASES = [                                             # ours: expected trajectories, not just expected answers
    {"question": "Hi there!",                                              "desk": "smalltalk", "tools": set()},
    {"question": "Can a failed course be retaken?",                        "desk": "faq",       "tools": set()},
    {"question": "How long can I keep a library book?",                    "desk": "faq",       "tools": set()},
    {"question": "What programme is student S003 on?",                     "desk": "records",   "tools": {"get_student"}},
    {"question": "Does CS201 have seats, and what does the handbook say about credits?", "desk": "records", "tools": {"get_course", "search_knowledge"}},
]

def evaluate(graph, cases):                                # ours: a small evaluation harness
    passed, total_ms, total_calls = 0, 0.0, 0
    for case in cases:
        started = time.perf_counter()
        out = graph.invoke({"messages": [HumanMessage(case["question"])]})   # LangGraph
        ms = 1000 * (time.perf_counter() - started)
        used = {c["name"] for m in out["messages"] if isinstance(m, AIMessage) for c in m.tool_calls}
        model_calls = sum(1 for m in out["messages"] if isinstance(m, AIMessage))
        ok = out["category"] == case["desk"] and case["tools"] <= used
        passed += ok; total_ms += ms; total_calls += model_calls
        print(f"  {'PASS' if ok else 'FAIL'}  desk={out['category']:9} tools={sorted(used) or '-'}  {case['question'][:55]}")
    print(f"\npass rate: {passed}/{len(cases)} | avg latency: {total_ms / len(cases):.0f} ms | avg model calls: {total_calls / len(cases):.1f}")

evaluate(campusai_v7, EVAL_CASES)

### Step 4 — A taste of LangSmith (optional, needs a free key)

LangSmith is the hosted tracing and evaluation product built for LangGraph. With two environment
variables set, every node, model call, tool call and token of every run is recorded, and each run
gets a URL you can open, share and comment on. Its evaluation side stores datasets like
`EVAL_CASES` and runs graders over them, which is the production version of Step 3.

To try it: create a free account at smith.langchain.com, create an API key, add it as a Colab
secret named `LANGSMITH_API_KEY`, and run the cell. Without a key the cell explains and moves on.

In [ ]:
def load_langsmith_key():                                   # ours: Colab secret, then environment
    try:
        from google.colab import userdata
        key = userdata.get("LANGSMITH_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.getenv("LANGSMITH_API_KEY", "")

LANGSMITH_KEY = load_langsmith_key()
if LANGSMITH_KEY:
    os.environ.update({"LANGSMITH_TRACING": "true", "LANGSMITH_API_KEY": LANGSMITH_KEY, "LANGSMITH_PROJECT": "campusai-langgraph-track"})   # that is all tracing needs
    out = campusai_v7.invoke({"messages": [HumanMessage("Does CS201 have seats, and what does the handbook say about credits?")]})
    from langsmith import Client                             # langsmith: the SDK
    time.sleep(3)                                            # traces are uploaded in the background
    runs = list(Client().list_runs(project_name="campusai-langgraph-track", is_root=True, limit=1))
    print("traced. open this run:", runs[0].url if runs else "(not uploaded yet; refresh the project page)")
    os.environ["LANGSMITH_TRACING"] = "false"                # keep the rest of the notebook untraced
else:
    print("No LANGSMITH_API_KEY found, so this run is not traced.")
    print("With a key, the same graph.invoke() produces a run page showing: triage -> records -> agent -> tools -> agent, each with inputs, outputs, latency and tokens.")

### Recap

- **Problem seen:** no progress feedback, no record of what ran, and no way to tell whether a change made the agent better or worse.
- **Layer added:** stream modes, a trace built from the updates stream, an evaluation harness over expected trajectories, and LangSmith tracing behind two environment variables.
- **Evidence:** nodes appeared as they finished; the trace listed every step with timings and tokens; five cases produced a pass rate.